In [1]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA
!mkdir -p /kaggle/working/outputs

In [ ]:
!git clone https://github.com/waterdisappear/SARATR-X.git
import sys
sys.path.append('/kaggle/working/SARATR-X/pre-training')

Cloning into 'SARATR-X'...
remote: Enumerating objects: 2322, done.
remote: Counting objects: 100% (2322/2322), done.
remote: Compressing objects: 100% (1645/1645), done.
remote: Total 2322 (delta 698), reused 2220 (delta 636), pack-reused 0 (from 0)
Receiving objects: 100% (2322/2322), 37.55 MiB | 23.51 MiB/s, done.
Resolving deltas: 100% (698/698), done.


In [3]:
!cp /kaggle/input/datasets/anamikapatel8/croma-base/CROMA_base.pt /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/pretrain_croma.py /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/use_croma.py /kaggle/working/CROMA/

In [4]:
!pip install rasterio
!pip install timm==0.5.4 huggingface_hub rasterio geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 10.7 MB/s eta 0:00:0000:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [5]:
import sys

sys.path.append("/kaggle/working/code")
sys.path.append("/kaggle/working/CROMA")

sys.path.append('/kaggle/working/SARATR-X/pre-training')
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# repo and path
checkpoint_path = hf_hub_download(
    repo_id="waterdisappear/SARATR-X",
    filename="mae_hivit_base_1600ep.pth",
    subfolder="pre-training", 
    token=hf_token
)
SARATR_CKPT = checkpoint_path

print(f"Success! Weights are at: {checkpoint_path}")

pre-training/mae_hivit_base_1600ep.pth:   0%|          | 0.00/263M [00:00<?, ?B/s]

Success! Weights are at: /root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth


In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import geopandas as gpd
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

PASTIS_ROOT = Path('/kaggle/input/datasets/anamikapatel8/pastis-r/PASTIS-R')
CROMA_CKPT  = Path('/kaggle/working/CROMA/CROMA_base.pt')


NUM_CLASSES   = 20       # 0=background, 1-18=crops, 19=void
IGNORE_INDEX  = 19
IMG_SIZE      = 128

# Shared token-grid alignment
# CROMA is patch_size=8, for 128x128: N = (128/8)^2 = 256 tokens.
# SARATR-X is patch_size=16, 64 tokens; SARATRXEncoder projects and
# bilinearly interpolates its 8x8 token grid up to 16x16 to match CROMA's grid
FUSION_PATCH_SIZE = 8
N_PATCHES = (IMG_SIZE // FUSION_PATCH_SIZE) ** 2  # = 256
ENCODER_DIM      = 768


SAR_CHANNELS = 3        # using S1A: [VV, VH, VV/VH]
SAR_SOURCE   = 'S1A'    


# PASTIS DATA_S2 has 10 bands: B2,B3,B4,B5,B6,B7,B8,B8A,B11,B12
# CROMA pretrained on 12 Sentinel-2 bands (B1..B12 minus B10)
# We pad 10 -> 12 by inserting zero channels at the B1 and B9 positions
S2_CHANNELS_DATA  = 10
S2_CHANNELS_CROMA = 12

# --- Number of dates to sample per patch ---
N_DATES_TO_USE = 12

BATCH_SIZE   = 4
EPOCHS       = 60
LR           = 1e-4
TRAIN_FOLDS  = [1, 2, 3]
VAL_FOLD     = [4]
TEST_FOLD    = [5]

print(f'N_PATCHES per encoder: {N_PATCHES}  (both CROMA and HiViT will produce this)')
print(f'SAR channels: {SAR_CHANNELS} (source: {SAR_SOURCE})')
print(f'S2 channels: {S2_CHANNELS_DATA} in data -> {S2_CHANNELS_CROMA} padded for CROMA')

Device: cuda
N_PATCHES per encoder: 256  (both CROMA and HiViT will produce this)
SAR channels: 3 (source: S1A)
S2 channels: 10 in data -> 12 padded for CROMA


In [ ]:
import torch.nn as nn

class S2LinearAdapter(nn.Module):
    """
    Adapts 10-channel PASTIS S2 data to 12 channels for CROMA using a 1x1 convolution.
    """
    def __init__(self):
        super().__init__()
        self.adapter = nn.Conv2d(S2_CHANNELS_DATA, S2_CHANNELS_CROMA, kernel_size=1)

    def forward(self, x):
        # x shape: (B, 10, H, W)
        return self.adapter(x)


class TemporalAttention(nn.Module):
    """
    Applies temporal attention across multiple dates for each patch location.
    Input: (B, n_dates, N_PATCHES, ENCODER_DIM)
    Output: (B, N_PATCHES, ENCODER_DIM)
    """
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attention_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu'
        )

    def forward(self, x):
        # x: (B, n_dates, N_PATCHES, ENCODER_DIM)
        B, n_dates, N_PATCHES, D = x.shape

        # Reshaping to (B * N_PATCHES, n_dates, ENCODER_DIM) for TransformerEncoderLayer
        # attention operates independently for each spatial patch across dates
        x_reshaped = x.permute(0, 2, 1, 3).reshape(B * N_PATCHES, n_dates, D)

        # Applying transformer encoder layer
        # output `attn_out`: (B * N_PATCHES, n_dates, ENCODER_DIM)
        attn_out = self.attention_layer(x_reshaped)

        # Aggregating across dates to get a single representation per patch
        # Output: (B * N_PATCHES, ENCODER_DIM)
        aggregated_patches = attn_out.mean(dim=1)

        # Reshaping back to (B, N_PATCHES, ENCODER_DIM)
        final_output = aggregated_patches.reshape(B, N_PATCHES, D)
        return final_output

In [ ]:
import random
import numpy as np
import torch
import torchvision.transforms as T 
import json
from pathlib import Path
import geopandas as gpd
from torch.utils.data import Dataset 
import torchvision.transforms.functional as TF 

def load_norm_stats(json_path, folds):
    with open(json_path) as f:
        stats = json.load(f)
    means = np.array([stats[f'Fold_{fold}']['mean'] for fold in folds], dtype=np.float32)
    stds  = np.array([stats[f'Fold_{fold}']['std']  for fold in folds], dtype=np.float32)
    return means.mean(axis=0), stds.mean(axis=0)

class PASTISFusionDataset(Dataset):
    def __init__(self, root, folds, norm=True, sar_source='S1A', n_dates=N_DATES_TO_USE,
                 augment=False, norm_stats=None):
        super().__init__()
        self.root = Path(root)
        self.norm = norm
        self.sar_source = sar_source
        self.n_dates = n_dates
        self.augment = augment

        meta = gpd.read_file(self.root / 'metadata.geojson')
        meta.index = meta['ID_PATCH'].astype(int)
        self.patches = meta[meta['Fold'].isin(folds)].index.tolist()

        if norm:
            if norm_stats is not None:
                # Using (train-fold) stats — required for val/test
                self.s2_mean, self.s2_std, self.s1a_mean, self.s1a_std = norm_stats
            else:
                # compute from this split's own folds 
                self.s2_mean, self.s2_std = load_norm_stats(self.root / 'NORM_S2_patch.json', folds)
                self.s1a_mean, self.s1a_std = load_norm_stats(self.root / 'NORM_S1A_patch.json', folds)

        if self.augment:
            self.photometric_transforms_s2 = T.Compose([
                T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)], p=0.5),
                T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.5),
            ])
        else:
            self.photometric_transforms_s2 = None


    def __len__(self):
        return len(self.patches)

    def _norm(self, arr, mean, std):
        return (arr - mean[None,:,None,None]) / (std[None,:,None,None] + 1e-6)

    def __getitem__(self, idx):
        pid = self.patches[idx]

        # 1. Loading stacks
        s2 = np.load(self.root / f'DATA_S2/S2_{pid}.npy').astype(np.float32) # (T, C, H, W)
        s1a = np.load(self.root / f'DATA_S1A/S1A_{pid}.npy').astype(np.float32) # (T, C, H, W)

        # 2.  Using N_DATES_TO_USE representative dates (early, mid, late)
        idx_s2 = np.linspace(0, s2.shape[0]-1, self.n_dates).astype(int)
        idx_sar = np.linspace(0, s1a.shape[0]-1, self.n_dates).astype(int)

        s2_selected = s2[idx_s2] # (n_dates, C, H, W)
        sar_selected = s1a[idx_sar] # (n_dates, C, H, W)

        if self.norm:
            s2_selected = self._norm(s2_selected, self.s2_mean, self.s2_std)
            sar_selected = self._norm(sar_selected, self.s1a_mean, self.s1a_std)

        label = np.load(self.root / f'ANNOTATIONS/TARGET_{pid}.npy')[0].astype(np.int64)
        label[label == 255] = 19

        # Converting to torch tensors early for consistent geometric augmentations
        s2_selected_t = torch.from_numpy(s2_selected) # (n_dates, 10, H, W)
        sar_selected_t = torch.from_numpy(sar_selected) # (n_dates, C, H, W)
        label_t = torch.from_numpy(label).unsqueeze(0) # (1, H, W) - ADDED UNSQUEEZE

        # 4. Augmentation (geometric first, then photometric)
        if self.augment:
            # Applying geometric transforms to all modalities 
            # Horizontal Flip
            if random.random() > 0.5:
                s2_selected_t = TF.hflip(s2_selected_t)
                sar_selected_t = TF.hflip(sar_selected_t)
                label_t = TF.hflip(label_t)
            # Vertical Flip
            if random.random() > 0.5:
                s2_selected_t = TF.vflip(s2_selected_t)
                sar_selected_t = TF.vflip(sar_selected_t)
                label_t = TF.vflip(label_t)
            # Random Rotation (-90, 0, 90, 180 degrees)
            if random.random() > 0.5:
                angle = random.choice([0, 90, 180, 270]) 
                s2_selected_t = TF.rotate(s2_selected_t, angle)
                sar_selected_t = TF.rotate(sar_selected_t, angle)
                label_t = TF.rotate(label_t, angle, interpolation=TF.InterpolationMode.NEAREST) # For segmentation masks

            # Photometric transforms for S2 only (applied per date)
            s2_stack_list = []
            for i in range(self.n_dates):
                s2_date_tensor = s2_selected_t[i] # (10, H, W)
                if self.photometric_transforms_s2:
                    # Extracting RGB bands (PASTIS band order: B2,B3,B4... -> indices 0,1,2...)
                    # For RGB display, use B4, B3, B2. So indices 2, 1, 0
                    rgb_bands = s2_date_tensor[[2, 1, 0], :, :].clone() # Shape (3, H, W)

                    # Applying photometric transforms to the 3-channel RGB
                    rgb_bands_transformed = self.photometric_transforms_s2(rgb_bands)

                    # Placing transformed RGB bands back into the 10-channel tensor
                    s2_date_tensor_modified = s2_date_tensor.clone() 
                    s2_date_tensor_modified[2, :, :] = rgb_bands_transformed[0, :, :] # Transformed B4 (Red)
                    s2_date_tensor_modified[1, :, :] = rgb_bands_transformed[1, :, :] # Transformed B3 (Green)
                    s2_date_tensor_modified[0, :, :] = rgb_bands_transformed[2, :, :] # Transformed B2 (Blue)

                    s2_date_tensor = s2_date_tensor_modified
                s2_stack_list.append(s2_date_tensor)
            # Concatenating dates along the channel dimension for the input to CROMAOpticalEncoder
            # (n_dates * 10, H, W)
            s2_stack = torch.cat(s2_stack_list, dim=0) # Result: (Dates * Channels, H, W)
            sar_stack = sar_selected_t.reshape(-1, 128, 128)
            label_tensor = label_t.squeeze(0) 

        else: # No augmentation
            # Concatenating dates along the channel dimension for the input to CROMAOpticalEncoder
            s2_stack = torch.cat([s for s in s2_selected_t], dim=0) # (n_dates * 10, H, W)
            sar_stack = sar_selected_t.reshape(-1, 128, 128) # Reshape SAR
            label_tensor = label_t.squeeze(0) # (H, W) 

        return s2_stack, sar_stack, label_tensor


# Building datasets 
train_s2_mean, train_s2_std = load_norm_stats(PASTIS_ROOT / 'NORM_S2_patch.json', TRAIN_FOLDS)
train_s1a_mean, train_s1a_std = load_norm_stats(PASTIS_ROOT / 'NORM_S1A_patch.json', TRAIN_FOLDS)
train_ds = PASTISFusionDataset(PASTIS_ROOT, TRAIN_FOLDS, sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE, norm_stats=(train_s2_mean, train_s2_std, train_s1a_mean, train_s1a_std), augment=True) # Enable augmentation
val_ds   = PASTISFusionDataset(PASTIS_ROOT, VAL_FOLD,   sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE,norm_stats=(train_s2_mean, train_s2_std, train_s1a_mean, train_s1a_std), augment=False)
test_ds  = PASTISFusionDataset(PASTIS_ROOT, TEST_FOLD,  sar_source=SAR_SOURCE, n_dates=N_DATES_TO_USE,norm_stats=(train_s2_mean, train_s2_std, train_s1a_mean, train_s1a_std), augment=False)

s2, sar, lbl = train_ds[0]
print(f'\nS2  : {s2.shape}   (10 bands, 128x128)')
print(f'SAR : {sar.shape}  ({SAR_CHANNELS} channels, 128x128)')
print(f'Label: {lbl.shape}  classes: {lbl.unique().tolist()}')


S2  : torch.Size([120, 128, 128])   (10 bands, 128x128)
SAR : torch.Size([36, 128, 128])  (3 channels, 128x128)
Label: torch.Size([128, 128])  classes: [0, 1, 2, 3, 4, 6, 12, 13, 19]


In [12]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

s2_b, sar_b, lbl_b = next(iter(train_loader))
print(f'Batch — S2: {s2_b.shape}  SAR: {sar_b.shape}  Label: {lbl_b.shape}')

Batch — S2: torch.Size([4, 120, 128, 128])  SAR: torch.Size([4, 36, 128, 128])  Label: torch.Size([4, 128, 128])


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/CROMA')

from use_croma import PretrainedCROMA
import torch
import torch.nn as nn
import torch.nn.functional as F

class CROMAOpticalEncoder(nn.Module):
    def __init__(self, ckpt_path, image_resolution=128, unfreeze_last_n_blocks=0, n_dates=3):
        super().__init__()
        self.n_dates = n_dates
        self.croma = PretrainedCROMA(
            pretrained_path=str(ckpt_path), size='base',
            modality='optical', image_resolution=image_resolution
        )
        for p in self.croma.parameters():
            p.requires_grad = False
        if unfreeze_last_n_blocks > 0:
            if hasattr(self.croma, '_model') and hasattr(self.croma._model, 'blocks') and isinstance(self.croma._model.blocks, nn.ModuleList):
                for i, block in enumerate(self.croma._model.blocks):
                    if i >= len(self.croma._model.blocks) - unfreeze_last_n_blocks:
                        for p in block.parameters():
                            p.requires_grad = True

        self.s2_adapter = S2LinearAdapter().to(DEVICE)
        self.temporal_attention = TemporalAttention(embed_dim=ENCODER_DIM, num_heads=8).to(DEVICE)
        print("CROMA Optical Encoder initialized with Temporal Attention.")

    def forward(self, x):
        B, C, H, W = x.shape
        s2_channels_per_date = S2_CHANNELS_DATA
        assert C == self.n_dates * s2_channels_per_date

        x_dates = torch.split(x, s2_channels_per_date, dim=1)
        encoded_dates = []
        for single_date_x in x_dates:
            adapted_single_date_x = self.s2_adapter(single_date_x)
            out = self.croma(optical_images=adapted_single_date_x)
            encoded_dates.append(out['optical_encodings'])

        stacked = torch.stack(encoded_dates, dim=1)      # (B, n_dates, N_PATCHES, D)
        return self.temporal_attention(stacked), None    # replaces mean pooling

In [ ]:
# #use it for temporal mean pooling
# import sys
# # Ensure the directory containing use_croma.py is in the Python path
# sys.path.insert(0, '/kaggle/working/CROMA')

# from use_croma import PretrainedCROMA
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class CROMAOpticalEncoder(nn.Module):
#     def __init__(self, ckpt_path, image_resolution=128, unfreeze_last_n_blocks=0, n_dates=3):
#         super().__init__()
#         self.n_dates = n_dates
#         # 1. Loads Pretrained CROMA
#         self.croma = PretrainedCROMA(
#             pretrained_path=str(ckpt_path),
#             size='base',
#             modality='optical',
#             image_resolution=image_resolution
#         )

#         # 2. Freezes parameters of CROMA initially
#         for p in self.croma.parameters():
#             p.requires_grad = False

#         # 3. Optionally unfreeze last 'n' blocks
#         if unfreeze_last_n_blocks > 0:
#             if hasattr(self.croma, '_model') and hasattr(self.croma._model, 'blocks') and isinstance(self.croma._model.blocks, nn.ModuleList):
#                 for i, block in enumerate(self.croma._model.blocks):
#                     if i >= len(self.croma._model.blocks) - unfreeze_last_n_blocks:
#                         for p in block.parameters():
#                             p.requires_grad = True
#                 print(f"CROMA Optical Encoder: Unfroze last {unfreeze_last_n_blocks} blocks.")

#         # S2 Band Mismatch with Linear Adapter ---
#         self.s2_adapter = S2LinearAdapter().to(DEVICE)

#         # Temporal aggregation via Mean Pooling (Replaces TemporalAttention) ---
#         print("CROMA Optical Encoder initialized with Temporal Mean Pooling.")

#     def forward(self, x):
#         B, C, H, W = x.shape
#         s2_channels_per_date = S2_CHANNELS_DATA

#         assert C == self.n_dates * s2_channels_per_date, \
#             f"CROMAOpticalEncoder expected {self.n_dates * s2_channels_per_date} channels, but got {C}"

#         x_dates = torch.split(x, s2_channels_per_date, dim=1)

#         encoded_dates = []
#         for single_date_x in x_dates:
#             adapted_single_date_x = self.s2_adapter(single_date_x)
#             out = self.croma(optical_images=adapted_single_date_x)
#             encoded_dates.append(out['optical_encodings']) # (B, N_PATCHES, ENCODER_DIM)

#         # Stacks encodings: (n_dates, B, N_PATCHES, ENCODER_DIM)
#         stacked_encodings = torch.stack(encoded_dates, dim=1)
        
#         # Temporal Mean Pooling: Average across the n_dates (dim 1)
#         fused_encodings = torch.mean(stacked_encodings, dim=1) # (B, N_PATCHES, ENCODER_DIM)

#         return fused_encodings, None

In [ ]:
class CrossModalFusion(nn.Module):
    """
    Enhanced Bidirectional cross-attention with a Transformer Refiner.
    Fixes stalling by allowing deeper feature interaction after the initial merge.
    """
    def __init__(self, dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.ms_to_sar = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.sar_to_ms = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_ms   = nn.LayerNorm(dim)
        self.norm_sar  = nn.LayerNorm(dim)

        # Learned gating to balance the two modalities
        self.gate      = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())

        # --- Transformer Refiner Layer ---
        # processes the fused tokens to ensure spatial consistency
        self.refiner   = nn.TransformerEncoderLayer(
            d_model=dim, nhead=num_heads, dim_feedforward=dim*4,
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.norm_out  = nn.LayerNorm(dim)
        self.drop      = nn.Dropout(dropout)

    def forward(self, ms_tokens, sar_tokens):
        # Shape guard
        assert ms_tokens.shape == sar_tokens.shape

        # 1. Bidirectional Cross-Attention
        ms_cross,  _ = self.ms_to_sar(ms_tokens,  sar_tokens, sar_tokens)
        sar_cross, _ = self.sar_to_ms(sar_tokens, ms_tokens,  ms_tokens)

        ms_out  = self.norm_ms(ms_tokens  + self.drop(ms_cross))
        sar_out = self.norm_sar(sar_tokens + self.drop(sar_cross))

        # 2. Gated Fusion
        gate   = self.gate(torch.cat([ms_out, sar_out], dim=-1))
        fused  = gate * ms_out + (1 - gate) * sar_out

        # 3. Refinement: Self-attention on fused representation
        refined = self.refiner(fused)

        return self.norm_out(refined)

class ConvSegHead(nn.Module):
    """
    Deepened Segmentation Head.
    Uses larger kernels (4x4) and residual-style blocks for better boundary definition.
    """
    def __init__(self, in_dim=768, num_classes=20, patch_size=8, img_size=128):
        super().__init__()
        self.h = self.w = img_size // patch_size   # 16

        self.head = nn.Sequential(
            # Projecting to lower dim 
            nn.Conv2d(in_dim, 512, 1),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.Dropout(0.3),

            # Upsampling 1: 16 -> 32
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.Dropout(0.3),

            # Upsampling 2: 32 -> 64
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.GELU(),

            # Upsample 3: 64 -> 128
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.GELU(),

            # Final refinement and classification
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, tokens):
        B, N, D = tokens.shape
        # Reshaping tokens back to spatial grid: (B, 768, 16, 16)
        x = tokens.permute(0, 2, 1).reshape(B, D, self.h, self.w)
        return self.head(x)

# Testing both modules together
import numpy as np
fusion_mod = CrossModalFusion(dim=ENCODER_DIM, num_heads=8).to(DEVICE)
seg_head   = ConvSegHead(in_dim=ENCODER_DIM, num_classes=NUM_CLASSES,
                         patch_size=FUSION_PATCH_SIZE, img_size=IMG_SIZE).to(DEVICE)

with torch.no_grad():
    ms_tok  = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    sar_tok = torch.randn(2, N_PATCHES, ENCODER_DIM).to(DEVICE)
    fused   = fusion_mod(ms_tok, sar_tok)
    logits  = seg_head(fused)
    print(f'Fusion output : {fused.shape}')
    print(f'Seg head output: {logits.shape}')
    assert logits.shape == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
    print('ALL shape assertions PASSED')

Fusion output : torch.Size([2, 256, 768])
Seg head output: torch.Size([2, 20, 128, 128])
ALL shape assertions PASSED


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

try:
    from models_hivit import hivit_base
    HIVIT_AVAILABLE = True
    print('HiViT (hivit_base) imported successfully from SARATR-X repo.')
except ImportError as e:
    HIVIT_AVAILABLE = False
    print(f'WARNING: HiViT import failed. Error: {e}')
    print('Using timm ViT fallback.')


class SARATRXEncoder(nn.Module):
    """
    SARATR-X SAR encoder, run natively at img_size=128 (real data resolution,
    patch_size=16 -> 8x8=64 tokens). We interpolate the pos_embed down to 8x8.
    """
    def __init__(self, ckpt_path, in_channels=3, img_size=128, unfreeze_last_n_blocks=0):
        super().__init__()
        self.pretrained_dim = 512
        self.img_size = img_size
        self.native_patch_size = 16
        self.native_grid = img_size // self.native_patch_size      # 128/16 = 8
        self.target_grid = int(N_PATCHES ** 0.5)                   # CROMA's grid, 16

        print(f'SAR encoder: NATIVE img_size={img_size} patch_size=16 -> '
              f'{self.native_grid}x{self.native_grid}={self.native_grid**2} tokens '
              f'(no input resizing), then interpolated to '
              f'{self.target_grid}x{self.target_grid}={N_PATCHES}')

        if HIVIT_AVAILABLE:
            self.backbone = hivit_base(img_size=img_size)   # build AT REAL resolution, 128

            checkpoint = torch.load(ckpt_path, map_location='cpu')
            state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint

            
            ckpt_pos_embed = state_dict.pop('absolute_pos_embed', None)

            msg = self.backbone.load_state_dict(state_dict, strict=False)
            print(f'Backbone loaded — missing={len(msg.missing_keys)}, '
                  f'unexpected={len(msg.unexpected_keys)}')

            
            if ckpt_pos_embed is not None:
                self._load_interpolated_pos_embed(ckpt_pos_embed)
            else:
                print('WARNING: no absolute_pos_embed found in checkpoint — '
                      'backbone is using its freshly initialized positional embedding.')
        else:
            self.backbone = timm.create_model(
                'vit_base_patch16_224', pretrained=False,
                img_size=img_size, in_chans=in_channels, num_classes=0,
            )
            self.pretrained_dim = self.backbone.embed_dim
            print(f'timm ViT fallback created. N_patches={self.native_grid**2}')

        self.dim_adapter = nn.Linear(self.pretrained_dim, 768)

        for p in self.backbone.parameters():
            p.requires_grad = False

        if unfreeze_last_n_blocks > 0 and hasattr(self.backbone, 'blocks') and isinstance(self.backbone.blocks, nn.ModuleList):
            for i, block in enumerate(self.backbone.blocks):
                if i >= len(self.backbone.blocks) - unfreeze_last_n_blocks:
                    for p in block.parameters():
                        p.requires_grad = True
            print(f'SARATR-X Encoder: unfroze last {unfreeze_last_n_blocks} blocks.')

    def _load_interpolated_pos_embed(self, ckpt_pos_embed):
        """Bicubically resample the checkpoint's pos_embed (native 14x14=196,
        saved at img_size=224) down to our real data's native grid (8x8=64)."""
        dim = ckpt_pos_embed.shape[-1]
        ckpt_grid = int(ckpt_pos_embed.shape[1] ** 0.5)   # 14

        pe = ckpt_pos_embed.reshape(1, ckpt_grid, ckpt_grid, dim).permute(0, 3, 1, 2)  # (1,D,14,14)
        pe = F.interpolate(pe, size=(self.native_grid, self.native_grid),
                            mode='bicubic', align_corners=False)
        pe = pe.permute(0, 2, 3, 1).reshape(1, self.native_grid ** 2, dim)  # (1,64,D)

        with torch.no_grad():
            self.backbone.absolute_pos_embed.data.copy_(pe)
        print(f'Interpolated absolute_pos_embed: {ckpt_grid}x{ckpt_grid} '
              f'-> {self.native_grid}x{self.native_grid} (bicubic)')

    def forward(self, x):
        # x is fed at (128x128)
        feats = self.backbone.forward_features(x)          # (B, 64, 512)
        B, N, _ = feats.shape
        assert N == self.native_grid ** 2, \
            f'SAR native token count mismatch: got {N}, expected {self.native_grid**2}.'

        feats = self.dim_adapter(feats)                     # (B, 64, 768)

        feats = feats.transpose(1, 2).reshape(B, 768, self.native_grid, self.native_grid)
        feats = F.interpolate(feats, size=(self.target_grid, self.target_grid),
                               mode='bilinear', align_corners=False)
        feats = feats.reshape(B, 768, -1).transpose(1, 2)   # (B, 256, 768)

        return feats

HiViT (hivit_base) imported successfully from SARATR-X repo.


In [ ]:
import torch.nn as nn
class SARATRSARProcessor(nn.Module):
    """
    Processes multi-date SAR input by splitting it into single dates,
    passing each through the SARATRXEncoder, and aggregating using Mean Pooling.
    """
    def __init__(self, saratr_ckpt, in_channels_per_date=3, img_size=128, n_dates=3, unfreeze_last_n_blocks=0):
        super().__init__()
        self.n_dates = n_dates
        self.sar_channels_per_date = in_channels_per_date
        self.sar_encoder = SARATRXEncoder(
            saratr_ckpt,
            in_channels=self.sar_channels_per_date,
            img_size=img_size,
            unfreeze_last_n_blocks=unfreeze_last_n_blocks,
        )

        for p in self.sar_encoder.parameters():
            if p.requires_grad:
                p.requires_grad = False

        if unfreeze_last_n_blocks > 0 and hasattr(self.sar_encoder.backbone, 'blocks') and isinstance(self.sar_encoder.backbone.blocks, nn.ModuleList):
            for i, block in enumerate(self.sar_encoder.backbone.blocks):
                if i >= len(self.sar_encoder.backbone.blocks) - unfreeze_last_n_blocks:
                    for p in block.parameters():
                        p.requires_grad = True

        for p in self.sar_encoder.dim_adapter.parameters():
            p.requires_grad = True
        
        self.temporal_attention = TemporalAttention(embed_dim=ENCODER_DIM, num_heads=8).to(DEVICE)

        print("SARATR-X SAR Processor initialized with Temporal Attention.")

    def forward(self, x):
        B, C, H, W = x.shape
        assert C == self.n_dates * self.sar_channels_per_date, \
            f"SARATRSARProcessor expected {self.n_dates * self.sar_channels_per_date} channels, but got {C}"

        x_dates = torch.split(x, self.sar_channels_per_date, dim=1)

        encoded_dates = []
        for single_date_x in x_dates:
            out = self.sar_encoder(single_date_x)
            encoded_dates.append(out) 
        
        stacked = torch.stack(encoded_dates, dim=1)      # (B, n_dates, N_PATCHES, D)
        return self.temporal_attention(stacked)          # (B, N_PATCHES, D)

class CROMAXFusionSegModel(nn.Module):
    """
    Enhanced Multi-Sensor Fusion Model for Semantic Segmentation.

    Architecture:
    1. S2 (10ch) -> CROMA Encoder -> MS Tokens (256, 768)
    2. SAR (3ch per date x 3 dates) -> SARATR-X Processor -> SAR Tokens (256, 768)
    3. FUSION: Bidirectional Cross-Attention + Transformer Refiner
    4. HEAD: Deep ConvTranspose Upsampling (16 -> 128)
    """
    def __init__(self, croma_ckpt, saratr_ckpt,
                 num_classes=20, img_size=128, in_sar_channels_total=9, n_dates=N_DATES_TO_USE, unfreeze_encoder_blocks=0):
        super().__init__()

        # 1. Encoders (Pre-trained Backbones)
        self.ms_encoder  = CROMAOpticalEncoder(croma_ckpt,  image_resolution=img_size, unfreeze_last_n_blocks=unfreeze_encoder_blocks, n_dates=n_dates)
        self.sar_encoder = SARATRSARProcessor(
            saratr_ckpt,
            in_channels_per_date=in_sar_channels_total // n_dates, #  3
            img_size=img_size,
            n_dates=n_dates,
            unfreeze_last_n_blocks=unfreeze_encoder_blocks # Pass unfreeze parameter
        )

        # 2. Deep Fusion (Cross-Attention + Self-Attention Refiner)
        self.fusion   = CrossModalFusion(dim=ENCODER_DIM, num_heads=8, dropout=0.1)

        # 3. Deep Segmentation Head (Enhanced Spatial Capacity)
        self.seg_head = ConvSegHead(in_dim=ENCODER_DIM, num_classes=num_classes,
                                    patch_size=FUSION_PATCH_SIZE, img_size=img_size)

    def forward(self, s2, sar):
        # Extracting features from both modalities
        ms_tokens, _  = self.ms_encoder(s2)     # (B, 256, 768)
        sar_tokens    = self.sar_encoder(sar)    # (B, 256, 768)
        # Fusing and Refining representation
        fused         = self.fusion(ms_tokens, sar_tokens)  # (B, 256, 768)
        # Upsampling to pixel-level logits
        return self.seg_head(fused)              # (B, 20, 128, 128)

    def trainable_params(self):
        return [
            {'params': (p for p in self.ms_encoder.parameters() if p.requires_grad), 'name': 'ms_encoder'},
            {'params': (p for p in self.sar_encoder.parameters() if p.requires_grad), 'name': 'sar_encoder'},
            {'params': (p for p in self.fusion.parameters() if p.requires_grad), 'name': 'fusion'},
            {'params': (p for p in self.seg_head.parameters() if p.requires_grad), 'name': 'seg_head'}
        ]

    def param_counts(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {'total': total, 'trainable': trainable, 'frozen': total - trainable}


# Instantiating model with unfreezing 2 last blocks of each encoder
model = CROMAXFusionSegModel(
    croma_ckpt      = CROMA_CKPT,
    saratr_ckpt     = SARATR_CKPT,
    num_classes     = NUM_CLASSES,
    img_size        = IMG_SIZE,
    in_sar_channels_total = N_DATES_TO_USE * SAR_CHANNELS, # total channels = n_dates * channels_per_date (12*3=36)
    n_dates         = N_DATES_TO_USE,
    unfreeze_encoder_blocks=0 #dont  Unfreeze
).to(DEVICE)

counts = model.param_counts()
print(f"Total    : {counts['total']:,}")
print(f"Trainable: {counts['trainable']:,}  <- fusion + seg head + unfrozen encoder blocks")
print(f"Frozen   : {counts['frozen']:,}  <- CROMA + SARATR-X (partially frozen)")

# Full end-to-end forward pass from real data
with torch.no_grad():
    s2_b, sar_b, lbl_b = next(iter(train_loader))
    logits = model(s2_b.to(DEVICE), sar_b.to(DEVICE))
    print(f'\nEnd-to-end forward: {s2_b.shape} + {sar_b.shape} -> {logits.shape}')
    assert logits.shape == (BATCH_SIZE, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
    print('Full forward pass PASSED')

Initializing optical encoder
CROMA Optical Encoder initialized with Temporal Attention.
SAR encoder: NATIVE img_size=128 patch_size=16 -> 8x8=64 tokens (no input resizing), then interpolated to 16x16=256


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Backbone loaded — missing=26, unexpected=3
Interpolated absolute_pos_embed: 14x14 -> 8x8 (bicubic)
SARATR-X SAR Processor initialized with Temporal Attention.
Total    : 187,352,000
Trainable: 30,738,360  <- fusion + seg head + unfrozen encoder blocks
Frozen   : 156,613,640  <- CROMA + SARATR-X (partially frozen)

End-to-end forward: torch.Size([4, 120, 128, 128]) + torch.Size([4, 36, 128, 128]) -> torch.Size([4, 20, 128, 128])
Full forward pass PASSED


In [ ]:
import numpy as np
import torch

# Pixel counts per class obtained from eval run (index = class id)
class_pixel_counts = {
    0: 3158992, 1: 1466003, 2: 560680, 3: 688289, 4: 163392,
    5: 129119, 6: 53762, 7: 81933, 8: 187477, 9: 71047,
    10: 58362, 11: 69960, 12: 60715, 13: 22613, 14: 127794,
    15: 68316, 16: 76859, 17: 42769, 18: 32024
}

freqs = np.array([class_pixel_counts[c] for c in range(19)], dtype=np.float64)
freqs = freqs / freqs.sum()
median_freq = np.median(freqs)

class_weights = median_freq / freqs          # median-frequency balancing
class_weights = np.clip(class_weights, 0.5, 10.0)   

# pad a 0 weight for the ignore_index slot (class 19) so the tensor is length NUM_CLASSES
full_weights = np.append(class_weights, 0.0)
class_weights_tensor = torch.tensor(full_weights, dtype=torch.float32).to(DEVICE)

print(dict(zip(range(20), full_weights.round(2))))

{0: np.float64(0.5), 1: np.float64(0.5), 2: np.float64(0.5), 3: np.float64(0.5), 4: np.float64(0.5), 5: np.float64(0.6), 6: np.float64(1.43), 7: np.float64(0.94), 8: np.float64(0.5), 9: np.float64(1.08), 10: np.float64(1.32), 11: np.float64(1.1), 12: np.float64(1.27), 13: np.float64(3.4), 14: np.float64(0.6), 15: np.float64(1.13), 16: np.float64(1.0), 17: np.float64(1.8), 18: np.float64(2.4), 19: np.float64(0.0)}


In [ ]:
import time
import sys
import numpy as np
import torch
import os

import torch.nn as nn
import torch.nn.functional as F

class PanopticMultiTaskLoss(nn.Module):
    def __init__(self, ignore_index=19, class_weights=None, ce_weight=1.0, dice_weight=1.0):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index, weight=class_weights)
        self.ignore_index = ignore_index
        self.dice_weights = None   
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def dice_loss(self, inputs, targets):
        smooth = 1e-6
        num_classes = inputs.shape[1]
        inputs = F.softmax(inputs, dim=1)
        valid_mask = (targets != self.ignore_index).float()
        total_dice_loss = 0.0
        total_weight = 0.0

        for cls in range(num_classes):
            if cls == self.ignore_index:
                continue
            w = self.dice_weights[cls].item() if self.dice_weights is not None else 1.0
            input_cls_masked = inputs[:, cls, :, :] * valid_mask
            target_cls_masked = (targets == cls).float() * valid_mask
            intersection = (input_cls_masked * target_cls_masked).sum()
            union = (input_cls_masked + target_cls_masked).sum()
            dice = (2. * intersection + smooth) / (union + smooth)
            total_dice_loss += w * (1 - dice)
            total_weight += w

        if total_weight == 0:
            return torch.tensor(0.0, device=inputs.device)
        return total_dice_loss / total_weight

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        d_loss = self.dice_loss(logits, targets)
        total = self.ce_weight * ce_loss + self.dice_weight * d_loss
        return total, ce_loss.detach(), d_loss.detach()   # return components too
        
# --- Metric Function ---
def compute_miou(preds, targets, num_classes=NUM_CLASSES, ignore_idx=IGNORE_INDEX):
    if preds.dim() == 4:
        preds = preds.argmax(dim=1)
    valid = targets != ignore_idx
    ious  = []
    for cls in range(num_classes):
        if cls == ignore_idx: continue
        p = (preds   == cls) & valid
        t = (targets == cls) & valid
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return float(np.mean(ious)) if ious else 0.0

# --- Setup & Initialization ---
CHECKPOINT_DIR = '/kaggle/working/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESUME_FROM_CHECKPOINT = '/kaggle/working/tis-latest/latest_checkpoint.pt' # '/content/drive/MyDrive/Major_Project/outputs/checkpoints/latest_checkpoint.pt' # Example: set this to resume

best_miu = 0.0
history = {'train_loss': [], 'train_ce': [], 'train_dice': [], 'val_loss': [], 'val_miou': []}
patience = 15
trigger_times = 0
start_epoch = 1

criterion = PanopticMultiTaskLoss(
    ignore_index=IGNORE_INDEX,
    class_weights=class_weights_tensor,   # from the median-frequency balancing
    ce_weight=1.0,
    dice_weight=2.0                       
).to(DEVICE)

model.to(DEVICE)

# optimizer = torch.optim.AdamW([
#     {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'ms_encoder' for p in p_group['params'] if p.requires_grad), 'lr': 5e-6, 'name': 'ms_encoder_unfrozen'},
#     {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'sar_encoder' for p in p_group['params'] if p.requires_grad), 'lr': 5e-6, 'name': 'sar_encoder_unfrozen'},
#     {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'fusion' for p in p_group['params'] if p.requires_grad), 'lr': 1e-4, 'name': 'fusion_module'},
#     {'params': (p for p_group in model.trainable_params() if p_group['name'] == 'seg_head' for p in p_group['params'] if p.requires_grad), 'lr': 1e-4, 'name': 'seg_head'}
# ], weight_decay=0.01)

new_modules = [model.ms_encoder.s2_adapter, model.ms_encoder.temporal_attention,
               model.sar_encoder.temporal_attention, model.sar_encoder.sar_encoder.dim_adapter]
new_params = [p for m in new_modules for p in m.parameters()]
new_ids = {id(p) for p in new_params}
pretrained_trainable = [p for p in list(model.ms_encoder.parameters()) + list(model.sar_encoder.parameters())
                        if p.requires_grad and id(p) not in new_ids]

groups = [
    {'params': new_params, 'lr': 1e-4, 'name': 'new_encoder_side_modules'},
    {'params': model.fusion.parameters(), 'lr': 1e-4, 'name': 'fusion_module'},
    {'params': model.seg_head.parameters(), 'lr': 1e-4, 'name': 'seg_head'},
]
if pretrained_trainable:   # only non-empty if you later unfreeze encoder blocks
    groups.append({'params': pretrained_trainable, 'lr': 5e-6, 'name': 'pretrained_unfrozen_blocks'})
optimizer = torch.optim.AdamW(groups, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=60, T_mult=1, eta_min=1e-6
)

# Loads checkpoint if specified
if RESUME_FROM_CHECKPOINT and os.path.exists(RESUME_FROM_CHECKPOINT):
    print(f"Resuming training from {RESUME_FROM_CHECKPOINT}...")
    checkpoint = torch.load(RESUME_FROM_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_miu = checkpoint['best_miu']
    history = checkpoint['history'] # Load history to continue plotting
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}, best mIoU: {best_miu:.4f}")
else:
    print("Starting training from scratch.")


# Initializes AMP scaler
scaler = torch.cuda.amp.GradScaler()
torch.cuda.empty_cache() # Clear cache before starting

print('Loss, optimiser, metrics ready.')
print(f"{'='*30}\nStarting Training on {DEVICE}\n{'='*30}")

for epoch in range(start_epoch, EPOCHS + 1):
    t_epoch_start = time.time()
    model.train()

    tr_loss = 0.0
    tr_ce = 0.0
    tr_dice = 0.0

    
    n_batches = len(train_loader)

    for batch_idx, (s2_b, sar_b, lbl_b) in enumerate(train_loader):
        s2_b, sar_b, lbl_b = s2_b.to(DEVICE), sar_b.to(DEVICE), lbl_b.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            logits = model(s2_b, sar_b)
            loss, ce_component, dice_component = criterion(logits, lbl_b)

        # Scaled backward pass for AMP
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for group in optimizer.param_groups for p in group['params'] if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()

        tr_loss += loss.item()
        tr_ce += ce_component.item()
        tr_dice += dice_component.item()

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            sys.stdout.write(f"\rEpoch {epoch:02d} | Batch {batch_idx+1}/{n_batches} | Loss: {tr_loss/(batch_idx+1):.4f}")
            sys.stdout.flush()


    history['train_loss'].append(tr_loss/len(train_loader))
    history['train_ce'].append(tr_ce/len(train_loader))
    history['train_dice'].append(tr_dice/len(train_loader))

    # Validation
    model.eval()
    val_intersections = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_unions = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_targets = torch.zeros(NUM_CLASSES, device=DEVICE)
    val_preds = torch.zeros(NUM_CLASSES, device=DEVICE)

    with torch.no_grad():
        for s2_b, sar_b, lbl_b in val_loader:
            s2_b, sar_b, lbl_b = s2_b.to(DEVICE), sar_b.to(DEVICE), lbl_b.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(s2_b, sar_b)
            preds = logits.argmax(dim=1)

            valid = lbl_b != IGNORE_INDEX
            for cls in range(NUM_CLASSES):
                if cls == IGNORE_INDEX: continue
                p = (preds == cls) & valid
                t = (lbl_b == cls) & valid
                val_intersections[cls] += (p & t).sum()
                val_unions[cls] += (p | t).sum()
                val_targets[cls] += t.sum()
                val_preds[cls] += p.sum()

    per_class_iou = val_intersections / (val_unions + 1e-6)
    per_class_dice = 2 * val_intersections / (val_preds + val_targets + 1e-6)

    valid_classes = [c for c in range(NUM_CLASSES) if c != IGNORE_INDEX and val_targets[c] > 0]
    va_miou = per_class_iou[valid_classes].mean().item()
    va_mdice = per_class_dice[valid_classes].mean().item()
    history['val_miou'].append(va_miou) # Store validation mIoU

    scheduler.step()

    # --- Checkpointing & Early Stopping ---
    checkpoint_state = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_miu': best_miu,
        'history': history
    }

    torch.save(checkpoint_state, os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pt'))

    if va_miou > best_miu:
        best_miu = va_miou
        trigger_times = 0
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_model.pt')) # Only model weights for easy deployment
        torch.save(checkpoint_state, os.path.join(CHECKPOINT_DIR, 'best_model_full_state.pt')) # Full state for best model
        improved_mark = " << BEST"
    else:
        trigger_times += 1
        improved_mark = f" (Patience: {trigger_times}/{patience})"

    print(f"\nEpoch {epoch:02d} | Loss: {tr_loss/len(train_loader):.4f} "
          f"(CE: {tr_ce/len(train_loader):.4f}, Dice: {tr_dice/len(train_loader):.4f}) | "
          f"Val mIoU: {va_miou:.4f} | Val mDice: {va_mdice:.4f} | Best: {best_miu:.4f}{improved_mark}")
    
    if trigger_times >= patience:
        print("Early stopping triggered!")
        break


Starting training from scratch.
Loss, optimiser, metrics ready.
Starting Training on cuda


/tmp/ipykernel_58/3089973013.py:129: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 01 | Batch 363/363 | Loss: 4.3136

/tmp/ipykernel_58/3089973013.py:184: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():



Epoch 01 | Loss: 4.3136 (CE: 2.4337, Dice: 0.9400) | Val mIoU: 0.1297 | Val mDice: 0.1821 | Best: 0.1297 << BEST
Epoch 02 | Batch 363/363 | Loss: 3.7725
Epoch 02 | Loss: 3.7725 (CE: 1.9822, Dice: 0.8951) | Val mIoU: 0.2020 | Val mDice: 0.2711 | Best: 0.2020 << BEST
Epoch 03 | Batch 363/363 | Loss: 3.4151
Epoch 03 | Loss: 3.4151 (CE: 1.6916, Dice: 0.8618) | Val mIoU: 0.2418 | Val mDice: 0.3240 | Best: 0.2418 << BEST
Epoch 04 | Batch 363/363 | Loss: 3.1698
Epoch 04 | Loss: 3.1698 (CE: 1.5010, Dice: 0.8344) | Val mIoU: 0.2649 | Val mDice: 0.3512 | Best: 0.2649 << BEST
Epoch 05 | Batch 363/363 | Loss: 3.0095
Epoch 05 | Loss: 3.0095 (CE: 1.3779, Dice: 0.8158) | Val mIoU: 0.2890 | Val mDice: 0.3810 | Best: 0.2890 << BEST
Epoch 06 | Batch 363/363 | Loss: 2.8383
Epoch 06 | Loss: 2.8383 (CE: 1.2544, Dice: 0.7920) | Val mIoU: 0.3407 | Val mDice: 0.4550 | Best: 0.3407 << BEST
Epoch 07 | Batch 363/363 | Loss: 2.7014
Epoch 07 | Loss: 2.7014 (CE: 1.1586, Dice: 0.7714) | Val mIoU: 0.3608 | Val mDice

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Loading the best model weights
state_dict = torch.load('/kaggle/working/best_model.pt', map_location=DEVICE)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=True)
model.eval()

print("Model loaded successfully!")
if len(missing_keys) > 0:
    print(f"Note: {len(missing_keys)} missing keys ignored.")
if len(unexpected_keys) > 0:
    print(f"Note: {len(unexpected_keys)} unexpected keys bypassed.")

# Initializing counters for global metric accumulation
total_intersections = torch.zeros(NUM_CLASSES, device=DEVICE)
total_unions = torch.zeros(NUM_CLASSES, device=DEVICE)
total_targets = torch.zeros(NUM_CLASSES, device=DEVICE)
total_preds = torch.zeros(NUM_CLASSES, device=DEVICE)

print("Starting evaluation...")

for batch_idx, (s2_val_b, sar_val_b, lbl_val_b) in enumerate(val_loader):
    s2_val_b, sar_val_b, lbl_val_b = s2_val_b.to(DEVICE), sar_val_b.to(DEVICE), lbl_val_b.to(DEVICE)

    with torch.no_grad():
        logits = model(s2_val_b, sar_val_b)
        preds = logits.argmax(dim=1)

    # Accumulating metrics per class globally
    valid = lbl_val_b != IGNORE_INDEX
    for cls in range(NUM_CLASSES):
        if cls == IGNORE_INDEX: continue
        p = (preds == cls) & valid
        t = (lbl_val_b == cls) & valid

        total_intersections[cls] += (p & t).sum()
        total_unions[cls] += (p | t).sum()
        total_targets[cls] += t.sum()
        total_preds[cls] += p.sum()

# PASTIS-R 18 Crops
CLASS_NAMES = {
    0: 'Background',
    1: 'Meadow',
    2: 'Soft winter wheat',
    3: 'Corn (Maize)',
    4: 'Winter barley',
    5: 'Winter rapeseed',
    6: 'Spring barley',
    7: 'Sunflower',
    8: 'Grapevine',
    9: 'Beet (Sugar beet)',
    10: 'Winter triticale',
    11: 'Winter durum wheat',
    12: 'Fruits, vegetables, and flowers',
    13: 'Potatoes',
    14: 'Leguminous fodder',
    15: 'Soybeans',
    16: 'Orchard',
    17: 'Mixed cereal',
    18: 'Sorghum',
    19: 'Void'
}

# Calculating final metrics
per_class_iou = total_intersections / (total_unions + 1e-6)
per_class_dice = 2 * total_intersections / (total_preds + total_targets + 1e-6)

print("\n--- Per-Class IoU & Dice Metrics ---")
valid_classes = []
for cls in range(NUM_CLASSES):
    if cls == IGNORE_INDEX: continue
    iou = per_class_iou[cls].item()
    dice = per_class_dice[cls].item()
    target_pixels = total_targets[cls].item()
    valid_classes.append(cls)
    print(f"Class {cls:02d} ({CLASS_NAMES.get(cls, 'Unknown')}): IoU = {iou:.4f}, Dice = {dice:.4f}, Pixels = {int(target_pixels)}")

final_miou = per_class_iou[valid_classes].mean().item()
print(f"\nOverall validation mIoU: {final_miou:.4f}")

Model loaded successfully!
Starting evaluation...

--- Per-Class IoU & Dice Metrics ---
Class 00 (Background): IoU = 0.6963, Dice = 0.8210, Pixels = 3158992
Class 01 (Meadow): IoU = 0.6110, Dice = 0.7585, Pixels = 1466003
Class 02 (Soft winter wheat): IoU = 0.7460, Dice = 0.8545, Pixels = 560680
Class 03 (Corn (Maize)): IoU = 0.7700, Dice = 0.8701, Pixels = 688289
Class 04 (Winter barley): IoU = 0.6730, Dice = 0.8045, Pixels = 163392
Class 05 (Winter rapeseed): IoU = 0.8046, Dice = 0.8917, Pixels = 129119
Class 06 (Spring barley): IoU = 0.3369, Dice = 0.5040, Pixels = 53762
Class 07 (Sunflower): IoU = 0.5125, Dice = 0.6777, Pixels = 81933
Class 08 (Grapevine): IoU = 0.4000, Dice = 0.5714, Pixels = 187477
Class 09 (Beet (Sugar beet)): IoU = 0.7921, Dice = 0.8840, Pixels = 71047
Class 10 (Winter triticale): IoU = 0.3627, Dice = 0.5323, Pixels = 58362
Class 11 (Winter durum wheat): IoU = 0.4728, Dice = 0.6421, Pixels = 69960
Class 12 (Fruits, vegetables, and flowers): IoU = 0.2853, Dice =

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------- settings (Qualitative results) ----------------
OUT_DIR    = '/kaggle/working/qualitative'
os.makedirs(OUT_DIR, exist_ok=True)
N_SAMPLES  = 4                          
DATE_IDX   = N_DATES_TO_USE // 2        
S2_BANDS   = (2, 1, 0)                  
USE_TTA    = False                      

PALETTE = (np.array(plt.cm.tab20.colors) * 255).astype(np.uint8)   # (20, 3)
PALETTE[IGNORE_INDEX] = (0, 0, 0)

def colorize(lbl):                                   # (H, W) int -> (H, W, 3)
    return PALETTE[lbl]

def stretch(img, lo=2, hi=98):                       
    out = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[-1]):
        a, b = np.percentile(img[..., c], (lo, hi))
        out[..., c] = np.clip((img[..., c] - a) / (b - a + 1e-6), 0, 1)
    return out

def s2_rgb(s2_sample):                               # (T*10, H, W) normalised tensor
    H, W = s2_sample.shape[-2:]
    x = s2_sample.reshape(N_DATES_TO_USE, S2_CHANNELS_DATA, H, W)[DATE_IDX].cpu().numpy()
    x = x * (train_s2_std[:, None, None] + 1e-6) + train_s2_mean[:, None, None]   
    return stretch(np.stack([x[b] for b in S2_BANDS], axis=-1))

def sar_vv(sar_sample):                              # (T*3, H, W) normalised tensor
    H, W = sar_sample.shape[-2:]
    x = sar_sample.reshape(N_DATES_TO_USE, SAR_CHANNELS, H, W)[DATE_IDX, 0].cpu().numpy()
    x = x * (train_s1a_std[0] + 1e-6) + train_s1a_mean[0]
    return stretch(x[..., None])[..., 0]

# ---------------- main loop ----------------
model.eval()
saved = 0
for batch_idx, (s2_b, sar_b, lbl_b) in enumerate(test_loader, start=1):
    with torch.no_grad():
        probs = predict_with_tta(model, s2_b.to(DEVICE), sar_b.to(DEVICE), use_tta=USE_TTA)
        preds = probs[:, :IGNORE_INDEX].argmax(1).cpu().numpy()   # skip the untrained void channel

    for i in range(s2_b.shape[0]):
        g, p  = lbl_b[i].numpy(), preds[i]
        valid = g != IGNORE_INDEX
        ious = []
        for c in np.unique(g[valid]):                              # per-sample mIoU over classes present
            pc, gc = (p == c) & valid, (g == c)
            ious.append((pc & gc).sum() / max((pc | gc).sum(), 1))
        miou_i = float(np.mean(ious))

        tag = f'Batch {batch_idx} Sample {i+1}'
        fig, ax = plt.subplots(2, 2, figsize=(6, 6))
        ax[0, 0].imshow(s2_rgb(s2_b[i]));                 ax[0, 0].set_title(f'{tag}\nS2 Input')
        ax[0, 1].imshow(colorize(g));                     ax[0, 1].set_title(f'{tag}\nGround Truth')
        ax[1, 0].imshow(sar_vv(sar_b[i]), cmap='gray');   ax[1, 0].set_title(f'{tag}\nSAR Input (VV)')
        ax[1, 1].imshow(colorize(p));                     ax[1, 1].set_title(f'{tag}\nPrediction (mIoU {miou_i:.2f})')
        for a in ax.ravel():
            a.axis('off')
        plt.tight_layout()
        plt.savefig(f'{OUT_DIR}/b{batch_idx}_s{i+1}.png', dpi=200, bbox_inches='tight')
        plt.show()

        saved += 1
        if saved >= N_SAMPLES:
            break
    if saved >= N_SAMPLES:
        break

handles = [mpatches.Patch(color=PALETTE[c] / 255, label=f'{c}: {CLASS_NAMES[c]}') for c in range(IGNORE_INDEX)]
handles.append(mpatches.Patch(color='black', label='Void (ignored)'))
fig = plt.figure(figsize=(10, 1.8))
fig.legend(handles=handles, loc='center', ncol=5, fontsize=7, frameon=False)
plt.savefig(f'{OUT_DIR}/legend.png', dpi=200, bbox_inches='tight')
plt.show()